In [1]:
# 01_hm_preprocessing.ipynb

import os
import json
import platform
from pathlib import Path

import numpy as np
import pandas as pd

from IPython.display import display

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 120)

print("Python:", platform.python_version())
print("Pandas:", pd.__version__)

Python: 3.12.12
Pandas: 2.3.3


In [2]:
import os
import platform
from pathlib import Path

def detect_environment():
    """
    Detect whether notebook is running in Kaggle, Colab, or local/VSCode.
    Kaggle is checked first because some Kaggle environments may still expose google.colab packages.
    """
    if Path("/kaggle/input").exists() or "KAGGLE_KERNEL_RUN_TYPE" in os.environ:
        return "kaggle"

    if "COLAB_RELEASE_TAG" in os.environ or "COLAB_GPU" in os.environ:
        return "colab"

    try:
        import google.colab  # noqa
        return "colab"
    except ImportError:
        return "local"


ENVIRONMENT = detect_environment()

print("Detected environment:", ENVIRONMENT)
print("Python:", platform.python_version())

if ENVIRONMENT == "colab":
    from google.colab import drive

    drive_root = Path("/content/drive/MyDrive")

    if not drive_root.exists():
        drive.mount("/content/drive")
    else:
        print("Google Drive already mounted.")

elif ENVIRONMENT == "kaggle":
    print("Kaggle detected. Dataset must be added through Kaggle 'Add Input'.")

else:
    print("Local / VSCode detected.")

Detected environment: kaggle
Python: 3.12.12
Kaggle detected. Dataset must be added through Kaggle 'Add Input'.


In [3]:
PROJECT_NAME = "hm-recommender"

REQUIRED_FILES = [
    "articles.csv",
    "customers.csv",
    "transactions_train.csv",
    "sample_submission.csv",
]

def has_required_files(folder: Path) -> bool:
    return all((folder / filename).exists() for filename in REQUIRED_FILES)


def find_hm_dataset_dir(search_roots, max_depth_recursive=True):
    """
    Dynamically find the folder containing the required H&M CSV files.
    First checks root folders directly.
    Then searches recursively.
    """
    search_roots = [Path(root) for root in search_roots if Path(root).exists()]

    print("Searching these roots:")
    for root in search_roots:
        print(" -", root)

    # 1. Direct check
    for root in search_roots:
        if has_required_files(root):
            return root

    # 2. Check immediate children first
    for root in search_roots:
        for child in root.iterdir():
            if child.is_dir() and has_required_files(child):
                return child

    # 3. Recursive fallback
    if max_depth_recursive:
        for root in search_roots:
            for articles_path in root.rglob("articles.csv"):
                candidate = articles_path.parent
                if has_required_files(candidate):
                    return candidate

    return None


# Optional manual override.
# Leave as None for automatic detection.
# Example:
# MANUAL_RAW_DATA_DIR = Path("/content/drive/MyDrive/H&M Dataset")
MANUAL_RAW_DATA_DIR = None


if ENVIRONMENT == "colab":
    PROJECT_DIR = Path("/content/drive/MyDrive") / PROJECT_NAME

    search_roots = [
        Path("/content/drive/MyDrive"),
        Path("/content"),
    ]

elif ENVIRONMENT == "kaggle":
    PROJECT_DIR = Path("/kaggle/working") / PROJECT_NAME

    search_roots = [
        Path("/kaggle/input"),
        Path("/kaggle/working"),
    ]

else:
    PROJECT_DIR = Path.cwd()

    search_roots = [
        Path.cwd(),
        Path.cwd() / "data",
        Path.cwd() / "data" / "raw",
        Path.home() / "Downloads",
        Path.home() / "Desktop",
        Path.home() / "Documents",
    ]


if MANUAL_RAW_DATA_DIR is not None:
    RAW_DATA_DIR = Path(MANUAL_RAW_DATA_DIR)

    if not has_required_files(RAW_DATA_DIR):
        raise FileNotFoundError(
            f"MANUAL_RAW_DATA_DIR was set to {RAW_DATA_DIR}, but it does not contain all required files."
        )
else:
    RAW_DATA_DIR = find_hm_dataset_dir(search_roots)

    if RAW_DATA_DIR is None:
        raise FileNotFoundError(
            "Could not automatically find the H&M dataset folder.\n\n"
            "Expected one folder to contain:\n"
            "- articles.csv\n"
            "- customers.csv\n"
            "- transactions_train.csv\n"
            "- sample_submission.csv\n\n"
            "If running in Kaggle, add the dataset using '+ Add Input'.\n"
            "If running in Colab, make sure your Google Drive is mounted and the files are inside MyDrive.\n"
            "If running locally, put the files in data/raw/ or set MANUAL_RAW_DATA_DIR."
        )


PROCESSED_DATA_DIR = PROJECT_DIR / "data" / "processed"
ARTIFACTS_DIR = PROJECT_DIR / "artifacts"

PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

print("\nFinal paths")
print("-" * 80)
print("ENVIRONMENT:", ENVIRONMENT)
print("RAW_DATA_DIR:", RAW_DATA_DIR)
print("PROJECT_DIR:", PROJECT_DIR)
print("PROCESSED_DATA_DIR:", PROCESSED_DATA_DIR)
print("ARTIFACTS_DIR:", ARTIFACTS_DIR)

print("\nRequired files found:")
for filename in REQUIRED_FILES:
    file_path = RAW_DATA_DIR / filename
    size_mb = file_path.stat().st_size / (1024 ** 2)
    print(f"{filename}: {size_mb:.2f} MB")

Searching these roots:
 - /kaggle/input
 - /kaggle/working

Final paths
--------------------------------------------------------------------------------
ENVIRONMENT: kaggle
RAW_DATA_DIR: /kaggle/input/competitions/h-and-m-personalized-fashion-recommendations
PROJECT_DIR: /kaggle/working/hm-recommender
PROCESSED_DATA_DIR: /kaggle/working/hm-recommender/data/processed
ARTIFACTS_DIR: /kaggle/working/hm-recommender/artifacts

Required files found:
articles.csv: 34.45 MB
customers.csv: 197.54 MB
transactions_train.csv: 3326.42 MB
sample_submission.csv: 257.76 MB


In [4]:
required_files = {
    "articles": "articles.csv",
    "customers": "customers.csv",
    "transactions": "transactions_train.csv",
    "sample_submission": "sample_submission.csv",
}

missing_files = []

for name, filename in required_files.items():
    file_path = RAW_DATA_DIR / filename
    if file_path.exists():
        size_mb = file_path.stat().st_size / (1024 ** 2)
        print(f"Found {filename}: {size_mb:.2f} MB")
    else:
        missing_files.append(str(file_path))

if missing_files:
    raise FileNotFoundError(
        "Missing required files:\n" + "\n".join(missing_files)
    )

Found articles.csv: 34.45 MB
Found customers.csv: 197.54 MB
Found transactions_train.csv: 3326.42 MB
Found sample_submission.csv: 257.76 MB


In [5]:
def memory_usage_mb(df: pd.DataFrame) -> float:
    return df.memory_usage(deep=True).sum() / (1024 ** 2)


def print_df_info(df: pd.DataFrame, name: str) -> None:
    print(f"\n{name}")
    print("-" * 80)
    print("Shape:", df.shape)
    print(f"Memory usage: {memory_usage_mb(df):.2f} MB")
    display(df.head())


def missing_value_report(df: pd.DataFrame, name: str) -> pd.DataFrame:
    report = (
        df.isna()
        .sum()
        .reset_index()
        .rename(columns={"index": "column", 0: "missing_count"})
    )
    report["missing_percent"] = (report["missing_count"] / len(df) * 100).round(4)
    report = report.sort_values("missing_count", ascending=False)

    print(f"\nMissing value report: {name}")
    display(report)

    return report


def normalize_article_id(series: pd.Series) -> pd.Series:
    """
    H&M article_id should be stored as a 10-character string.
    Example: 108775015 -> 0108775015
    """
    return (
        series.astype("string")
        .str.replace(".0", "", regex=False)
        .str.strip()
        .str.zfill(10)
    )


def normalize_customer_id(series: pd.Series) -> pd.Series:
    return series.astype("string").str.strip()


def save_parquet(df: pd.DataFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_parquet(path, index=False, compression="snappy")
    size_mb = path.stat().st_size / (1024 ** 2)
    print(f"Saved: {path.name} | {size_mb:.2f} MB")

In [6]:
articles_path = RAW_DATA_DIR / "articles.csv"
customers_path = RAW_DATA_DIR / "customers.csv"
transactions_path = RAW_DATA_DIR / "transactions_train.csv"
sample_submission_path = RAW_DATA_DIR / "sample_submission.csv"

articles = pd.read_csv(
    articles_path,
    dtype={"article_id": "string"}
)

customers = pd.read_csv(
    customers_path,
    dtype={"customer_id": "string", "postal_code": "string"}
)

transactions = pd.read_csv(
    transactions_path,
    dtype={
        "customer_id": "string",
        "article_id": "string",
        "price": "float32",
        "sales_channel_id": "int8",
    },
    parse_dates=["t_dat"]
)

sample_submission = pd.read_csv(
    sample_submission_path,
    dtype={"customer_id": "string", "prediction": "string"}
)

print_df_info(articles, "Raw articles")
print_df_info(customers, "Raw customers")
print_df_info(transactions, "Raw transactions")
print_df_info(sample_submission, "Raw sample_submission")


Raw articles
--------------------------------------------------------------------------------
Shape: (105542, 25)
Memory usage: 111.38 MB


,article_id,product_code,prod_name,product_type_no,product_type_name,product_group_name,graphical_appearance_no,graphical_appearance_name,colour_group_code,colour_group_name,perceived_colour_value_id,perceived_colour_value_name,perceived_colour_master_id,perceived_colour_master_name,department_no,department_name,index_code,index_name,index_group_no,index_group_name,section_no,section_name,garment_group_no,garment_group_name,detail_desc
0,0108775015,108775,Strap top,253,Vest top,Garment Upper body,1010016,Solid,9,Black,4,Dark,5,Black,1676,Jersey Basic,A,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.
1,0108775044,108775,Strap top,253,Vest top,Garment Upper body,1010016,Solid,10,White,3,Light,9,White,1676,Jersey Basic,A,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.
2,0108775051,108775,Strap top (1),253,Vest top,Garment Upper body,1010017,Stripe,11,Off White,1,Dusty Light,9,White,1676,Jersey Basic,A,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.
3,0110065001,110065,OP T-shirt (Idro),306,Bra,Underwear,1010016,Solid,9,Black,4,Dark,5,Black,1339,Clean Lingerie,B,Lingeries/Tights,1,Ladieswear,61,Womens Lingerie,1017,"Under-, Nightwear","Microfibre T-shirt bra with underwired, moulde..."
4,0110065002,110065,OP T-shirt (Idro),306,Bra,Underwear,1010016,Solid,10,White,3,Light,9,White,1339,Clean Lingerie,B,Lingeries/Tights,1,Ladieswear,61,Womens Lingerie,1017,"Under-, Nightwear","Microfibre T-shirt bra with underwired, moulde..."



Raw customers
--------------------------------------------------------------------------------
Shape: (1371980, 7)
Memory usage: 470.60 MB


,customer_id,FN,Active,club_member_status,fashion_news_frequency,age,postal_code
0,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,NaN,NaN,ACTIVE,NONE,49.0,52043ee2162cf5aa7ee79974281641c6f11a68d276429a...
1,0000423b00ade91418cceaf3b26c6af3dd342b51fd051e...,NaN,NaN,ACTIVE,NONE,25.0,2973abc54daa8a5f8ccfe9362140c63247c5eee03f1d93...
2,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,NaN,NaN,ACTIVE,NONE,24.0,64f17e6a330a85798e4998f62d0930d14db8db1c054af6...
3,00005ca1c9ed5f5146b52ac8639a40ca9d57aeff4d1bd2...,NaN,NaN,ACTIVE,NONE,54.0,5d36574f52495e81f019b680c843c443bd343d5ca5b1c2...
4,00006413d8573cd20ed7128e53b7b13819fe5cfc2d801f...,1.0,1.0,ACTIVE,Regularly,52.0,25fa5ddee9aac01b35208d01736e57942317d756b32ddd...



Raw transactions
--------------------------------------------------------------------------------
Shape: (31788324, 5)
Memory usage: 5608.41 MB


,t_dat,customer_id,article_id,price,sales_channel_id
0,2018-09-20,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,0663713001,0.050831,2
1,2018-09-20,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,0541518023,0.030492,2
2,2018-09-20,00007d2de826758b65a93dd24ce629ed66842531df6699...,0505221004,0.015237,2
3,2018-09-20,00007d2de826758b65a93dd24ce629ed66842531df6699...,0685687003,0.016932,2
4,2018-09-20,00007d2de826758b65a93dd24ce629ed66842531df6699...,0685687004,0.016932,2



Raw sample_submission
--------------------------------------------------------------------------------
Shape: (1371980, 2)
Memory usage: 383.37 MB


,customer_id,prediction
0,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,0706016001 0706016002 0372860001 0610776002 07...
1,0000423b00ade91418cceaf3b26c6af3dd342b51fd051e...,0706016001 0706016002 0372860001 0610776002 07...
2,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,0706016001 0706016002 0372860001 0610776002 07...
3,00005ca1c9ed5f5146b52ac8639a40ca9d57aeff4d1bd2...,0706016001 0706016002 0372860001 0610776002 07...
4,00006413d8573cd20ed7128e53b7b13819fe5cfc2d801f...,0706016001 0706016002 0372860001 0610776002 07...


In [7]:
# Standardize IDs
articles["article_id"] = normalize_article_id(articles["article_id"])
transactions["article_id"] = normalize_article_id(transactions["article_id"])

customers["customer_id"] = normalize_customer_id(customers["customer_id"])
transactions["customer_id"] = normalize_customer_id(transactions["customer_id"])
sample_submission["customer_id"] = normalize_customer_id(sample_submission["customer_id"])

# Clean customer columns
if "FN" in customers.columns:
    customers["FN"] = customers["FN"].fillna(0).astype("int8")

if "Active" in customers.columns:
    customers["Active"] = customers["Active"].fillna(0).astype("int8")

if "age" in customers.columns:
    customers["age"] = pd.to_numeric(customers["age"], errors="coerce").astype("Int16")

if "club_member_status" in customers.columns:
    customers["club_member_status"] = (
        customers["club_member_status"]
        .astype("string")
        .fillna("UNKNOWN")
        .str.upper()
    )

if "fashion_news_frequency" in customers.columns:
    customers["fashion_news_frequency"] = (
        customers["fashion_news_frequency"]
        .astype("string")
        .fillna("UNKNOWN")
        .str.upper()
        .replace({"NONE": "NONE", "NAN": "UNKNOWN"})
    )

# Clean article text columns
for col in articles.columns:
    if articles[col].dtype == "object" or str(articles[col].dtype) == "string":
        if col != "article_id":
            articles[col] = articles[col].astype("string").fillna("UNKNOWN")

if "detail_desc" in articles.columns:
    articles["detail_desc"] = (
        articles["detail_desc"]
        .astype("string")
        .fillna("")
    )

# Make sure transaction values are valid
transactions["price"] = pd.to_numeric(transactions["price"], errors="coerce").astype("float32")
transactions["sales_channel_id"] = transactions["sales_channel_id"].astype("int8")

print_df_info(articles, "Cleaned articles")
print_df_info(customers, "Cleaned customers")
print_df_info(transactions, "Cleaned transactions")


Cleaned articles
--------------------------------------------------------------------------------
Shape: (105542, 25)
Memory usage: 111.39 MB


,article_id,product_code,prod_name,product_type_no,product_type_name,product_group_name,graphical_appearance_no,graphical_appearance_name,colour_group_code,colour_group_name,perceived_colour_value_id,perceived_colour_value_name,perceived_colour_master_id,perceived_colour_master_name,department_no,department_name,index_code,index_name,index_group_no,index_group_name,section_no,section_name,garment_group_no,garment_group_name,detail_desc
0,0108775015,108775,Strap top,253,Vest top,Garment Upper body,1010016,Solid,9,Black,4,Dark,5,Black,1676,Jersey Basic,A,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.
1,0108775044,108775,Strap top,253,Vest top,Garment Upper body,1010016,Solid,10,White,3,Light,9,White,1676,Jersey Basic,A,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.
2,0108775051,108775,Strap top (1),253,Vest top,Garment Upper body,1010017,Stripe,11,Off White,1,Dusty Light,9,White,1676,Jersey Basic,A,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.
3,0110065001,110065,OP T-shirt (Idro),306,Bra,Underwear,1010016,Solid,9,Black,4,Dark,5,Black,1339,Clean Lingerie,B,Lingeries/Tights,1,Ladieswear,61,Womens Lingerie,1017,"Under-, Nightwear","Microfibre T-shirt bra with underwired, moulde..."
4,0110065002,110065,OP T-shirt (Idro),306,Bra,Underwear,1010016,Solid,10,White,3,Light,9,White,1339,Clean Lingerie,B,Lingeries/Tights,1,Ladieswear,61,Womens Lingerie,1017,"Under-, Nightwear","Microfibre T-shirt bra with underwired, moulde..."



Cleaned customers
--------------------------------------------------------------------------------
Shape: (1371980, 7)
Memory usage: 446.24 MB


,customer_id,FN,Active,club_member_status,fashion_news_frequency,age,postal_code
0,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,0,0,ACTIVE,NONE,49,52043ee2162cf5aa7ee79974281641c6f11a68d276429a...
1,0000423b00ade91418cceaf3b26c6af3dd342b51fd051e...,0,0,ACTIVE,NONE,25,2973abc54daa8a5f8ccfe9362140c63247c5eee03f1d93...
2,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,0,0,ACTIVE,NONE,24,64f17e6a330a85798e4998f62d0930d14db8db1c054af6...
3,00005ca1c9ed5f5146b52ac8639a40ca9d57aeff4d1bd2...,0,0,ACTIVE,NONE,54,5d36574f52495e81f019b680c843c443bd343d5ca5b1c2...
4,00006413d8573cd20ed7128e53b7b13819fe5cfc2d801f...,1,1,ACTIVE,REGULARLY,52,25fa5ddee9aac01b35208d01736e57942317d756b32ddd...



Cleaned transactions
--------------------------------------------------------------------------------
Shape: (31788324, 5)
Memory usage: 5608.41 MB


,t_dat,customer_id,article_id,price,sales_channel_id
0,2018-09-20,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,0663713001,0.050831,2
1,2018-09-20,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,0541518023,0.030492,2
2,2018-09-20,00007d2de826758b65a93dd24ce629ed66842531df6699...,0505221004,0.015237,2
3,2018-09-20,00007d2de826758b65a93dd24ce629ed66842531df6699...,0685687003,0.016932,2
4,2018-09-20,00007d2de826758b65a93dd24ce629ed66842531df6699...,0685687004,0.016932,2


In [8]:
audit = {}

audit["articles_rows"] = int(len(articles))
audit["customers_rows"] = int(len(customers))
audit["transactions_rows"] = int(len(transactions))
audit["sample_submission_rows"] = int(len(sample_submission))

audit["unique_articles_in_articles"] = int(articles["article_id"].nunique())
audit["unique_customers_in_customers"] = int(customers["customer_id"].nunique())
audit["unique_articles_in_transactions"] = int(transactions["article_id"].nunique())
audit["unique_customers_in_transactions"] = int(transactions["customer_id"].nunique())

audit["transaction_start_date"] = str(transactions["t_dat"].min().date())
audit["transaction_end_date"] = str(transactions["t_dat"].max().date())

audit["duplicate_transaction_rows"] = int(transactions.duplicated().sum())

print(json.dumps(audit, indent=2))

{
  "articles_rows": 105542,
  "customers_rows": 1371980,
  "transactions_rows": 31788324,
  "sample_submission_rows": 1371980,
  "unique_articles_in_articles": 105542,
  "unique_customers_in_customers": 1371980,
  "unique_articles_in_transactions": 104547,
  "unique_customers_in_transactions": 1362281,
  "transaction_start_date": "2018-09-20",
  "transaction_end_date": "2020-09-22",
  "duplicate_transaction_rows": 2974905
}


In [9]:
articles_missing = missing_value_report(articles, "articles")
customers_missing = missing_value_report(customers, "customers")
transactions_missing = missing_value_report(transactions, "transactions")
sample_submission_missing = missing_value_report(sample_submission, "sample_submission")


Missing value report: articles


,column,missing_count,missing_percent
0,article_id,0,0.0
1,product_code,0,0.0
2,prod_name,0,0.0
3,product_type_no,0,0.0
4,product_type_name,0,0.0
5,product_group_name,0,0.0
6,graphical_appearance_no,0,0.0
7,graphical_appearance_name,0,0.0
8,colour_group_code,0,0.0
9,colour_group_name,0,0.0



Missing value report: customers


,column,missing_count,missing_percent
5,age,15861,1.1561
1,FN,0,0.0000
0,customer_id,0,0.0000
2,Active,0,0.0000
3,club_member_status,0,0.0000
4,fashion_news_frequency,0,0.0000
6,postal_code,0,0.0000



Missing value report: transactions


,column,missing_count,missing_percent
0,t_dat,0,0.0
1,customer_id,0,0.0
2,article_id,0,0.0
3,price,0,0.0
4,sales_channel_id,0,0.0



Missing value report: sample_submission


,column,missing_count,missing_percent
0,customer_id,0,0.0
1,prediction,0,0.0


In [10]:
# Build mappings from the union of metadata tables and transaction tables.
# This prevents missing IDs if a transaction contains an ID not present in articles/customers.

all_customer_ids = pd.concat(
    [
        customers["customer_id"],
        transactions["customer_id"],
        sample_submission["customer_id"],
    ],
    ignore_index=True,
).dropna().drop_duplicates().sort_values().reset_index(drop=True)

all_article_ids = pd.concat(
    [
        articles["article_id"],
        transactions["article_id"],
    ],
    ignore_index=True,
).dropna().drop_duplicates().sort_values().reset_index(drop=True)

customer_id_map = pd.DataFrame({
    "customer_id": all_customer_ids,
    "customer_idx": np.arange(len(all_customer_ids), dtype=np.int32)
})

article_id_map = pd.DataFrame({
    "article_id": all_article_ids,
    "article_idx": np.arange(len(all_article_ids), dtype=np.int32)
})

print_df_info(customer_id_map, "Customer ID map")
print_df_info(article_id_map, "Article ID map")


Customer ID map
--------------------------------------------------------------------------------
Shape: (1371980, 2)
Memory usage: 153.09 MB


,customer_id,customer_idx
0,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,0
1,0000423b00ade91418cceaf3b26c6af3dd342b51fd051e...,1
2,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,2
3,00005ca1c9ed5f5146b52ac8639a40ca9d57aeff4d1bd2...,3
4,00006413d8573cd20ed7128e53b7b13819fe5cfc2d801f...,4



Article ID map
--------------------------------------------------------------------------------
Shape: (105542, 2)
Memory usage: 6.34 MB


,article_id,article_idx
0,0108775015,0
1,0108775044,1
2,0108775051,2
3,0110065001,3
4,0110065002,4


In [11]:
transactions_processed = transactions.merge(
    customer_id_map,
    on="customer_id",
    how="left",
    validate="many_to_one"
)

transactions_processed = transactions_processed.merge(
    article_id_map,
    on="article_id",
    how="left",
    validate="many_to_one"
)

articles_processed = articles.merge(
    article_id_map,
    on="article_id",
    how="left",
    validate="one_to_one"
)

customers_processed = customers.merge(
    customer_id_map,
    on="customer_id",
    how="left",
    validate="one_to_one"
)

sample_submission_processed = sample_submission.merge(
    customer_id_map,
    on="customer_id",
    how="left",
    validate="many_to_one"
)

# Safety checks
assert transactions_processed["customer_idx"].isna().sum() == 0
assert transactions_processed["article_idx"].isna().sum() == 0
assert articles_processed["article_idx"].isna().sum() == 0
assert customers_processed["customer_idx"].isna().sum() == 0

transactions_processed["customer_idx"] = transactions_processed["customer_idx"].astype("int32")
transactions_processed["article_idx"] = transactions_processed["article_idx"].astype("int32")
articles_processed["article_idx"] = articles_processed["article_idx"].astype("int32")
customers_processed["customer_idx"] = customers_processed["customer_idx"].astype("int32")

print_df_info(transactions_processed, "Transactions with integer IDs")
print_df_info(articles_processed, "Articles with integer IDs")
print_df_info(customers_processed, "Customers with integer IDs")


Transactions with integer IDs
--------------------------------------------------------------------------------
Shape: (31788324, 7)
Memory usage: 5850.93 MB


,t_dat,customer_id,article_id,price,sales_channel_id,customer_idx,article_idx
0,2018-09-20,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,0663713001,0.050831,2,2,40179
1,2018-09-20,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,0541518023,0.030492,2,2,10520
2,2018-09-20,00007d2de826758b65a93dd24ce629ed66842531df6699...,0505221004,0.015237,2,7,6387
3,2018-09-20,00007d2de826758b65a93dd24ce629ed66842531df6699...,0685687003,0.016932,2,7,46304
4,2018-09-20,00007d2de826758b65a93dd24ce629ed66842531df6699...,0685687004,0.016932,2,7,46305



Articles with integer IDs
--------------------------------------------------------------------------------
Shape: (105542, 26)
Memory usage: 111.79 MB


,article_id,product_code,prod_name,product_type_no,product_type_name,product_group_name,graphical_appearance_no,graphical_appearance_name,colour_group_code,colour_group_name,perceived_colour_value_id,perceived_colour_value_name,perceived_colour_master_id,perceived_colour_master_name,department_no,department_name,index_code,index_name,index_group_no,index_group_name,section_no,section_name,garment_group_no,garment_group_name,detail_desc,article_idx
0,0108775015,108775,Strap top,253,Vest top,Garment Upper body,1010016,Solid,9,Black,4,Dark,5,Black,1676,Jersey Basic,A,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.,0
1,0108775044,108775,Strap top,253,Vest top,Garment Upper body,1010016,Solid,10,White,3,Light,9,White,1676,Jersey Basic,A,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.,1
2,0108775051,108775,Strap top (1),253,Vest top,Garment Upper body,1010017,Stripe,11,Off White,1,Dusty Light,9,White,1676,Jersey Basic,A,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.,2
3,0110065001,110065,OP T-shirt (Idro),306,Bra,Underwear,1010016,Solid,9,Black,4,Dark,5,Black,1339,Clean Lingerie,B,Lingeries/Tights,1,Ladieswear,61,Womens Lingerie,1017,"Under-, Nightwear","Microfibre T-shirt bra with underwired, moulde...",3
4,0110065002,110065,OP T-shirt (Idro),306,Bra,Underwear,1010016,Solid,10,White,3,Light,9,White,1339,Clean Lingerie,B,Lingeries/Tights,1,Ladieswear,61,Womens Lingerie,1017,"Under-, Nightwear","Microfibre T-shirt bra with underwired, moulde...",4



Customers with integer IDs
--------------------------------------------------------------------------------
Shape: (1371980, 8)
Memory usage: 451.48 MB


,customer_id,FN,Active,club_member_status,fashion_news_frequency,age,postal_code,customer_idx
0,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,0,0,ACTIVE,NONE,49,52043ee2162cf5aa7ee79974281641c6f11a68d276429a...,0
1,0000423b00ade91418cceaf3b26c6af3dd342b51fd051e...,0,0,ACTIVE,NONE,25,2973abc54daa8a5f8ccfe9362140c63247c5eee03f1d93...,1
2,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,0,0,ACTIVE,NONE,24,64f17e6a330a85798e4998f62d0930d14db8db1c054af6...,2
3,00005ca1c9ed5f5146b52ac8639a40ca9d57aeff4d1bd2...,0,0,ACTIVE,NONE,54,5d36574f52495e81f019b680c843c443bd343d5ca5b1c2...,3
4,00006413d8573cd20ed7128e53b7b13819fe5cfc2d801f...,1,1,ACTIVE,REGULARLY,52,25fa5ddee9aac01b35208d01736e57942317d756b32ddd...,4


In [12]:
# Event-level implicit feedback table.
# Each row = one purchase event.
# ALS later can aggregate repeated purchases per customer-item pair.

interactions = transactions_processed[
    [
        "t_dat",
        "customer_idx",
        "article_idx",
        "price",
        "sales_channel_id",
    ]
].copy()

interactions["event_type"] = "purchase"
interactions["weight"] = np.float32(1.0)

interactions = interactions.sort_values("t_dat").reset_index(drop=True)

print_df_info(interactions, "Model-ready interactions")


Model-ready interactions
--------------------------------------------------------------------------------
Shape: (31788324, 7)
Memory usage: 2485.89 MB


,t_dat,customer_idx,article_idx,price,sales_channel_id,event_type,weight
0,2018-09-20,2,40179,0.050831,2,purchase,1.0
1,2018-09-20,904365,16334,0.042356,2,purchase,1.0
2,2018-09-20,904365,28076,0.025407,2,purchase,1.0
3,2018-09-20,904427,20130,0.016949,1,purchase,1.0
4,2018-09-20,904427,20129,0.033898,1,purchase,1.0


In [13]:
interactions_aggregated = (
    interactions
    .groupby(["customer_idx", "article_idx"], as_index=False)
    .agg(
        purchase_count=("weight", "sum"),
        first_purchase_date=("t_dat", "min"),
        last_purchase_date=("t_dat", "max"),
        avg_price=("price", "mean"),
    )
)

interactions_aggregated["purchase_count"] = interactions_aggregated["purchase_count"].astype("float32")
interactions_aggregated["avg_price"] = interactions_aggregated["avg_price"].astype("float32")

# Simple implicit confidence weight.
# Base purchase weight starts at 1.0, and repeated purchases increase confidence.
# This aligns with the project plan before ALS tuning.
interactions_aggregated["implicit_weight"] = (
    1.0 + np.log1p(interactions_aggregated["purchase_count"])
).astype("float32")

print_df_info(interactions_aggregated, "Aggregated customer-item interactions")


Aggregated customer-item interactions
--------------------------------------------------------------------------------
Shape: (27306439, 7)
Memory usage: 937.49 MB


,customer_idx,article_idx,purchase_count,first_purchase_date,last_purchase_date,avg_price,implicit_weight
0,0,99,1.0,2018-12-27,2018-12-27,0.035576,1.693147
1,0,16003,2.0,2019-05-25,2019-05-25,0.050831,2.098612
2,0,16023,1.0,2020-09-05,2020-09-05,0.050831,1.693147
3,0,23996,1.0,2019-07-25,2019-07-25,0.012695,1.693147
4,0,29516,1.0,2018-12-27,2018-12-27,0.044051,1.693147


In [14]:
# Chronological split:
# earliest 70% interactions = train
# next 15% = validation
# latest 15% = test

interactions = interactions.sort_values("t_dat").reset_index(drop=True)

n = len(interactions)

train_boundary_position = int(n * 0.70)
val_boundary_position = int(n * 0.85)

train_end_date = interactions.loc[train_boundary_position, "t_dat"]
val_end_date = interactions.loc[val_boundary_position, "t_dat"]

train_interactions = interactions[interactions["t_dat"] < train_end_date].copy()
val_interactions = interactions[
    (interactions["t_dat"] >= train_end_date) &
    (interactions["t_dat"] < val_end_date)
].copy()
test_interactions = interactions[interactions["t_dat"] >= val_end_date].copy()

split_audit = {
    "total_interactions": int(len(interactions)),
    "train_rows": int(len(train_interactions)),
    "val_rows": int(len(val_interactions)),
    "test_rows": int(len(test_interactions)),
    "train_percent": round(len(train_interactions) / len(interactions) * 100, 2),
    "val_percent": round(len(val_interactions) / len(interactions) * 100, 2),
    "test_percent": round(len(test_interactions) / len(interactions) * 100, 2),
    "train_start_date": str(train_interactions["t_dat"].min().date()),
    "train_end_date": str(train_interactions["t_dat"].max().date()),
    "val_start_date": str(val_interactions["t_dat"].min().date()),
    "val_end_date": str(val_interactions["t_dat"].max().date()),
    "test_start_date": str(test_interactions["t_dat"].min().date()),
    "test_end_date": str(test_interactions["t_dat"].max().date()),
}

print(json.dumps(split_audit, indent=2))

{
  "total_interactions": 31788324,
  "train_rows": 22235277,
  "val_rows": 4760953,
  "test_rows": 4792094,
  "train_percent": 69.95,
  "val_percent": 14.98,
  "test_percent": 15.08,
  "train_start_date": "2018-09-20",
  "train_end_date": "2020-02-10",
  "val_start_date": "2020-02-11",
  "val_end_date": "2020-06-08",
  "test_start_date": "2020-06-09",
  "test_end_date": "2020-09-22"
}


In [15]:
train_users = set(train_interactions["customer_idx"].unique())
val_users = set(val_interactions["customer_idx"].unique())
test_users = set(test_interactions["customer_idx"].unique())

train_items = set(train_interactions["article_idx"].unique())
val_items = set(val_interactions["article_idx"].unique())
test_items = set(test_interactions["article_idx"].unique())

overlap_audit = {
    "train_unique_users": len(train_users),
    "val_unique_users": len(val_users),
    "test_unique_users": len(test_users),

    "train_unique_items": len(train_items),
    "val_unique_items": len(val_items),
    "test_unique_items": len(test_items),

    "val_users_seen_in_train_percent": round(len(val_users & train_users) / max(len(val_users), 1) * 100, 2),
    "test_users_seen_in_train_percent": round(len(test_users & train_users) / max(len(test_users), 1) * 100, 2),

    "val_items_seen_in_train_percent": round(len(val_items & train_items) / max(len(val_items), 1) * 100, 2),
    "test_items_seen_in_train_percent": round(len(test_items & train_items) / max(len(test_items), 1) * 100, 2),

    "cold_start_val_users": len(val_users - train_users),
    "cold_start_test_users": len(test_users - train_users),

    "cold_start_val_items": len(val_items - train_items),
    "cold_start_test_items": len(test_items - train_items),
}

print(json.dumps(overlap_audit, indent=2))

{
  "train_unique_users": 1150557,
  "val_unique_users": 574129,
  "test_unique_users": 578785,
  "train_unique_items": 83275,
  "val_unique_items": 43449,
  "test_unique_items": 44902,
  "val_users_seen_in_train_percent": 78.51,
  "test_users_seen_in_train_percent": 78.79,
  "val_items_seen_in_train_percent": 74.75,
  "test_items_seen_in_train_percent": 56.65,
  "cold_start_val_users": 123404,
  "cold_start_test_users": 122751,
  "cold_start_val_items": 10971,
  "cold_start_test_items": 19466
}


In [16]:
save_parquet(articles_processed, PROCESSED_DATA_DIR / "articles_processed.parquet")
save_parquet(customers_processed, PROCESSED_DATA_DIR / "customers_processed.parquet")
save_parquet(sample_submission_processed, PROCESSED_DATA_DIR / "sample_submission_processed.parquet")

save_parquet(customer_id_map, PROCESSED_DATA_DIR / "customer_id_map.parquet")
save_parquet(article_id_map, PROCESSED_DATA_DIR / "article_id_map.parquet")

save_parquet(transactions_processed, PROCESSED_DATA_DIR / "transactions_processed.parquet")
save_parquet(interactions, PROCESSED_DATA_DIR / "interactions.parquet")
save_parquet(interactions_aggregated, PROCESSED_DATA_DIR / "interactions_aggregated.parquet")

save_parquet(train_interactions, PROCESSED_DATA_DIR / "train_interactions.parquet")
save_parquet(val_interactions, PROCESSED_DATA_DIR / "val_interactions.parquet")
save_parquet(test_interactions, PROCESSED_DATA_DIR / "test_interactions.parquet")

Saved: articles_processed.parquet | 6.66 MB
Saved: customers_processed.parquet | 164.73 MB
Saved: sample_submission_processed.parquet | 84.86 MB
Saved: customer_id_map.parquet | 84.84 MB
Saved: article_id_map.parquet | 1.20 MB
Saved: transactions_processed.parquet | 873.25 MB
Saved: interactions.parquet | 191.21 MB
Saved: interactions_aggregated.parquet | 191.97 MB
Saved: train_interactions.parquet | 134.36 MB
Saved: val_interactions.parquet | 28.11 MB
Saved: test_interactions.parquet | 28.72 MB


In [17]:
# Detect environment safely (works even if kernel restarted)
ENVIRONMENT = os.environ.get("ENVIRONMENT", None)

if ENVIRONMENT is None:
    if Path("/kaggle/input").exists() or "KAGGLE_KERNEL_RUN_TYPE" in os.environ:
        ENVIRONMENT = "kaggle"
    elif "COLAB_RELEASE_TAG" in os.environ or "COLAB_GPU" in os.environ:
        ENVIRONMENT = "colab"
    else:
        try:
            import google.colab  # noqa
            ENVIRONMENT = "colab"
        except ImportError:
            ENVIRONMENT = "local"

IN_COLAB = ENVIRONMENT == "colab"
IN_KAGGLE = ENVIRONMENT == "kaggle"

# Build audit report
full_audit_report = {
    "environment": {
        "environment": ENVIRONMENT,
        "in_colab": IN_COLAB,
        "in_kaggle": IN_KAGGLE,
        "raw_data_dir": str(RAW_DATA_DIR),
        "project_dir": str(PROJECT_DIR),
        "processed_data_dir": str(PROCESSED_DATA_DIR),
    },
    "raw_dataset_audit": audit,
    "split_audit": split_audit,
    "overlap_audit": overlap_audit,
}

audit_path = ARTIFACTS_DIR / "hm_preprocessing_audit.json"

with open(audit_path, "w") as f:
    json.dump(full_audit_report, f, indent=2)

print(f"Saved audit report: {audit_path}")

Saved audit report: /kaggle/working/hm-recommender/artifacts/hm_preprocessing_audit.json


In [18]:
saved_files = sorted(PROCESSED_DATA_DIR.glob("*.parquet"))

saved_report = []

for file in saved_files:
    saved_report.append({
        "file": file.name,
        "size_mb": round(file.stat().st_size / (1024 ** 2), 2)
    })

saved_report_df = pd.DataFrame(saved_report)
display(saved_report_df)

print("Total processed files:", len(saved_files))

,file,size_mb
0,article_id_map.parquet,1.20
1,articles_processed.parquet,6.66
2,customer_id_map.parquet,84.84
3,customers_processed.parquet,164.73
4,interactions.parquet,191.21
5,interactions_aggregated.parquet,191.97
6,sample_submission_processed.parquet,84.86
7,test_interactions.parquet,28.72
8,train_interactions.parquet,134.36
9,transactions_processed.parquet,873.25


Total processed files: 11


In [19]:
# Make sure the saved files can actually be loaded again.

reload_train = pd.read_parquet(PROCESSED_DATA_DIR / "train_interactions.parquet")
reload_articles = pd.read_parquet(PROCESSED_DATA_DIR / "articles_processed.parquet")
reload_customers = pd.read_parquet(PROCESSED_DATA_DIR / "customers_processed.parquet")

print_df_info(reload_train, "Reloaded train_interactions")
print_df_info(reload_articles, "Reloaded articles_processed")
print_df_info(reload_customers, "Reloaded customers_processed")


Reloaded train_interactions
--------------------------------------------------------------------------------
Shape: (22235277, 7)
Memory usage: 1738.83 MB


,t_dat,customer_idx,article_idx,price,sales_channel_id,event_type,weight
0,2018-09-20,2,40179,0.050831,2,purchase,1.0
1,2018-09-20,7,46302,0.016932,2,purchase,1.0
2,2018-09-20,7,6386,0.020322,2,purchase,1.0
3,2018-09-20,198,47416,0.030492,1,purchase,1.0
4,2018-09-20,198,5944,0.053373,1,purchase,1.0



Reloaded articles_processed
--------------------------------------------------------------------------------
Shape: (105542, 26)
Memory usage: 111.79 MB


,article_id,product_code,prod_name,product_type_no,product_type_name,product_group_name,graphical_appearance_no,graphical_appearance_name,colour_group_code,colour_group_name,perceived_colour_value_id,perceived_colour_value_name,perceived_colour_master_id,perceived_colour_master_name,department_no,department_name,index_code,index_name,index_group_no,index_group_name,section_no,section_name,garment_group_no,garment_group_name,detail_desc,article_idx
0,0108775015,108775,Strap top,253,Vest top,Garment Upper body,1010016,Solid,9,Black,4,Dark,5,Black,1676,Jersey Basic,A,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.,0
1,0108775044,108775,Strap top,253,Vest top,Garment Upper body,1010016,Solid,10,White,3,Light,9,White,1676,Jersey Basic,A,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.,1
2,0108775051,108775,Strap top (1),253,Vest top,Garment Upper body,1010017,Stripe,11,Off White,1,Dusty Light,9,White,1676,Jersey Basic,A,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.,2
3,0110065001,110065,OP T-shirt (Idro),306,Bra,Underwear,1010016,Solid,9,Black,4,Dark,5,Black,1339,Clean Lingerie,B,Lingeries/Tights,1,Ladieswear,61,Womens Lingerie,1017,"Under-, Nightwear","Microfibre T-shirt bra with underwired, moulde...",3
4,0110065002,110065,OP T-shirt (Idro),306,Bra,Underwear,1010016,Solid,10,White,3,Light,9,White,1339,Clean Lingerie,B,Lingeries/Tights,1,Ladieswear,61,Womens Lingerie,1017,"Under-, Nightwear","Microfibre T-shirt bra with underwired, moulde...",4



Reloaded customers_processed
--------------------------------------------------------------------------------
Shape: (1371980, 8)
Memory usage: 451.48 MB


,customer_id,FN,Active,club_member_status,fashion_news_frequency,age,postal_code,customer_idx
0,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,0,0,ACTIVE,NONE,49,52043ee2162cf5aa7ee79974281641c6f11a68d276429a...,0
1,0000423b00ade91418cceaf3b26c6af3dd342b51fd051e...,0,0,ACTIVE,NONE,25,2973abc54daa8a5f8ccfe9362140c63247c5eee03f1d93...,1
2,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,0,0,ACTIVE,NONE,24,64f17e6a330a85798e4998f62d0930d14db8db1c054af6...,2
3,00005ca1c9ed5f5146b52ac8639a40ca9d57aeff4d1bd2...,0,0,ACTIVE,NONE,54,5d36574f52495e81f019b680c843c443bd343d5ca5b1c2...,3
4,00006413d8573cd20ed7128e53b7b13819fe5cfc2d801f...,1,1,ACTIVE,REGULARLY,52,25fa5ddee9aac01b35208d01736e57942317d756b32ddd...,4


In [20]:
print("H&M preprocessing completed successfully.")
print()
print("Main files for next steps:")
print(PROCESSED_DATA_DIR / "train_interactions.parquet")
print(PROCESSED_DATA_DIR / "val_interactions.parquet")
print(PROCESSED_DATA_DIR / "test_interactions.parquet")
print(PROCESSED_DATA_DIR / "articles_processed.parquet")
print(PROCESSED_DATA_DIR / "customers_processed.parquet")
print(PROCESSED_DATA_DIR / "customer_id_map.parquet")
print(PROCESSED_DATA_DIR / "article_id_map.parquet")
print()
print("Next notebook: 02_hm_eda.ipynb")

H&M preprocessing completed successfully.

Main files for next steps:
/kaggle/working/hm-recommender/data/processed/train_interactions.parquet
/kaggle/working/hm-recommender/data/processed/val_interactions.parquet
/kaggle/working/hm-recommender/data/processed/test_interactions.parquet
/kaggle/working/hm-recommender/data/processed/articles_processed.parquet
/kaggle/working/hm-recommender/data/processed/customers_processed.parquet
/kaggle/working/hm-recommender/data/processed/customer_id_map.parquet
/kaggle/working/hm-recommender/data/processed/article_id_map.parquet

Next notebook: 02_hm_eda.ipynb


In [21]:
def aggregate_interactions_for_model(df: pd.DataFrame, name: str) -> pd.DataFrame:
    aggregated = (
        df
        .groupby(["customer_idx", "article_idx"], as_index=False)
        .agg(
            purchase_count=("weight", "sum"),
            first_purchase_date=("t_dat", "min"),
            last_purchase_date=("t_dat", "max"),
            avg_price=("price", "mean"),
        )
    )

    aggregated["purchase_count"] = aggregated["purchase_count"].astype("float32")
    aggregated["avg_price"] = aggregated["avg_price"].astype("float32")
    aggregated["implicit_weight"] = (
        1.0 + np.log1p(aggregated["purchase_count"])
    ).astype("float32")

    print_df_info(aggregated, f"{name} aggregated interactions")
    return aggregated


train_interactions_aggregated = aggregate_interactions_for_model(
    train_interactions,
    "Train"
)

val_interactions_aggregated = aggregate_interactions_for_model(
    val_interactions,
    "Validation"
)

test_interactions_aggregated = aggregate_interactions_for_model(
    test_interactions,
    "Test"
)

save_parquet(
    train_interactions_aggregated,
    PROCESSED_DATA_DIR / "train_interactions_aggregated.parquet"
)

save_parquet(
    val_interactions_aggregated,
    PROCESSED_DATA_DIR / "val_interactions_aggregated.parquet"
)

save_parquet(
    test_interactions_aggregated,
    PROCESSED_DATA_DIR / "test_interactions_aggregated.parquet"
)


Train aggregated interactions
--------------------------------------------------------------------------------
Shape: (19106319, 7)
Memory usage: 655.96 MB


,customer_idx,article_idx,purchase_count,first_purchase_date,last_purchase_date,avg_price,implicit_weight
0,0,99,1.0,2018-12-27,2018-12-27,0.035576,1.693147
1,0,16003,2.0,2019-05-25,2019-05-25,0.050831,2.098612
2,0,23996,1.0,2019-07-25,2019-07-25,0.012695,1.693147
3,0,29516,1.0,2018-12-27,2018-12-27,0.044051,1.693147
4,0,30327,1.0,2018-12-27,2018-12-27,0.030492,1.693147



Validation aggregated interactions
--------------------------------------------------------------------------------
Shape: (4106191, 7)
Memory usage: 140.98 MB


,customer_idx,article_idx,purchase_count,first_purchase_date,last_purchase_date,avg_price,implicit_weight
0,0,78719,1.0,2020-03-21,2020-03-21,0.014390,1.693147
1,0,90060,1.0,2020-03-21,2020-03-21,0.011508,1.693147
2,0,93744,1.0,2020-03-21,2020-03-21,0.014390,1.693147
3,0,99926,1.0,2020-03-21,2020-03-21,0.021593,1.693147
4,0,100484,1.0,2020-03-21,2020-03-21,0.031763,1.693147



Test aggregated interactions
--------------------------------------------------------------------------------
Shape: (4222886, 7)
Memory usage: 144.98 MB


,customer_idx,article_idx,purchase_count,first_purchase_date,last_purchase_date,avg_price,implicit_weight
0,0,16023,1.0,2020-09-05,2020-09-05,0.050831,1.693147
1,1,87145,1.0,2020-07-08,2020-07-08,0.027102,1.693147
2,2,78503,1.0,2020-09-15,2020-09-15,0.061000,1.693147
3,4,60763,1.0,2020-08-12,2020-08-12,0.033881,1.693147
4,4,77915,1.0,2020-08-12,2020-08-12,0.020322,1.693147


Saved: train_interactions_aggregated.parquet | 129.63 MB
Saved: val_interactions_aggregated.parquet | 28.65 MB
Saved: test_interactions_aggregated.parquet | 29.17 MB
